In [ ]:
import json
import os
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm

sys.path.append('..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
)
from utils.analysis_utils import extract_verdict

# Constants
MAIN_AGENT_MODELS = ['claude-3-7-sonnet-20250219']
MONITOR_AGENT_MODELS = ['gpt-4o']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]

# Cache for combined_results
_combined_results_cache = {}


In [ ]:
def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        return _combined_results_cache[cache_key]
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    _combined_results_cache[cache_key] = combined_results
    return combined_results

def prepare_system_prompt_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    index = base_dir.split('_')[-1]
    system_prompt_df = load_monitoring_data(base_dir=base_dir)
    system_prompt_df[f'response_{index}'] = system_prompt_df.data.apply(lambda x: x['response_1']['content'])
    system_prompt_df[f'suspiciousness_{index}'] = system_prompt_df.data.apply(lambda x: extract_verdict(x['response_1']['content'], tag_word='verdict'))
    system_prompt_df[f'usage_{index}'] = system_prompt_df.data.apply(lambda x: x['response_1']['usage'])
    return system_prompt_df

def merge_and_compute_features_for_system_prompt(combined_results: pd.DataFrame, 
                             system_prompt_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        system_prompt_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()

    # Compute features
    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))

    return merged_df

In [ ]:
def load_and_merge_all_system_prompts(include_failed_attempts=False, include_failed_main_task=False):
    """
    Load all system_prompt dataframes (1-9) and merge them on common columns.
    
    Args:
        include_failed_attempts: Whether to include failed attempts
        include_failed_main_task: Whether to include failed main tasks
        
    Returns:
        pd.DataFrame: Merged dataframe containing data from all system prompts
    """
    # Load all system prompt dataframes
    system_prompt_dfs = {}
    for i in range(1, 10):  # 1 to 9 inclusive
        base_dir = f'monitor_responses/system_prompt_{i}'
        system_prompt_df = prepare_system_prompt_data(base_dir=base_dir, 
                                       include_failed_attempts=include_failed_attempts, 
                                       include_failed_main_task=include_failed_main_task)
        combined_results = load_combined_results(include_failed_attempts=include_failed_attempts, 
                                       include_failed_main_task=include_failed_main_task)
        df = merge_and_compute_features_for_system_prompt(combined_results, system_prompt_df)
        # Drop the data column before merging to avoid duplicate column issues
        if 'data' in df.columns:
            df = df.drop(columns=['data', 'default_suspiciousness', 'main_task_evaluation', 'main_task_success', 'num_assistant_messages', 'num_failed_tool_calls', 'combined_score', 'convo_length'])
        system_prompt_dfs[i] = df
        print(f"Loaded system_prompt_{i} with shape {df.shape}")
    
    # Start with the first dataframe
    merged_df = system_prompt_dfs[1]
    
    # Merge with the rest of the dataframes
    for i in range(2, 10):
        merged_df = pd.merge(
            merged_df,
            system_prompt_dfs[i],
            on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 'side_task_evaluation'],
            how='inner',
            suffixes=(f'_{i-1}', f'_{i}')  # Use explicit suffixes to avoid conflicts
        )
        print(f"After merging system_prompt_{i}, shape: {merged_df.shape}")
    
    return merged_df

def get_roc_curve(df, metric):
    true_labels = (df.side_task_evaluation == 1).astype(int)
    scores = df[metric] / 100
    fpr, tpr, _ = roc_curve(true_labels, scores)
    return fpr, tpr

def interpolate_tpr(fpr_points, tpr_points, target_fpr):
    """
    Interpolate the TPR value for a given FPR value using linear interpolation.
    
    Args:
        fpr_points: Array of false positive rates from ROC curve
        tpr_points: Array of true positive rates from ROC curve
        target_fpr: The FPR value for which to find the corresponding TPR
        
    Returns:
        Interpolated TPR value
    """
    # Handle edge cases
    if target_fpr <= fpr_points[0]:
        return tpr_points[0]
    if target_fpr >= fpr_points[-1]:
        return tpr_points[-1]
    
    # Find the two closest FPR points for interpolation
    for i in range(len(fpr_points) - 1):
        if fpr_points[i] <= target_fpr <= fpr_points[i + 1]:
            # Linear interpolation formula
            slope = (tpr_points[i + 1] - tpr_points[i]) / (fpr_points[i + 1] - fpr_points[i])
            return tpr_points[i] + slope * (target_fpr - fpr_points[i])
    
    # This should not happen if the edge cases are handled correctly
    return None

# Example usage:
combined_results = load_combined_results(include_failed_attempts=False, include_failed_main_task=False)

see_the_scratchpad_df = prepare_system_prompt_data(base_dir='monitor_responses/see_the_scratchpad', include_failed_attempts=False, include_failed_main_task=False)
merged_df = merge_and_compute_features_for_system_prompt(combined_results, see_the_scratchpad_df)

In [ ]:
plot_roc_curves(merged_df, combined_results, metrics=['suspiciousness_scratchpad'], give_hint=False, include_failed_attempts=False, include_failed_main_task=False, monitors=['gpt-4o'])
plot_suspiciousness_distributions(merged_df, metrics=['suspiciousness_scratchpad'], monitors=['gpt-4o'])

In [ ]:
print(merged_df[merged_df.transcript_type != 'transcripts_no_secondary_task'].sort_values(by='suspiciousness_scratchpad', ascending=True).iloc[0].response_scratchpad)

In [ ]:
fpr, tpr, thresholds = roc_curve(merged_df[merged_df.transcript_type != 'transcripts'].transcript_type != 'transcripts_no_secondary_task', merged_df[merged_df.transcript_type !='transcripts'].suspiciousness_scratchpad)
auc_score = auc(fpr, tpr)
print(f"AUC score: {auc_score}")

